# Lorenz96 Sliding-Window 4D-Var Benchmark

This notebook implements and evaluates the cycling (sliding-window) 4D-Var assimilation strategy for the Lorenz96 system. It compares the performance of a full-window optimization against a continuous cycling approach.

In [ ]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial

# Ensure project root is in path
if '..' not in sys.path:
    sys.path.append('..')

import pytorch_invobs_lib as pil
from pytorch_invobs_lib import (
    Lorenz96, generate_data, estimate_correlation, 
    baseline_init_l96, run_4dvar_l96
)

device = torch.device('cpu')
pil.device = device
print(f"Using device: {device}")

## 1. Sliding Window Logic
The function below implements the cycling mechanism where the analysis of one window is forecasted forward to serve as the background state for the next.

In [ ]:
def run_sliding_window_4dvar_l96(
    dyn_sys, inverter, corr, Y_long,
    window_T=10,
    stride=1,
    sigma_b=1.0,
    sigma_obs=0.5,
    sigma_p=0.5,
    init_mode='baseline',
    opt_mode='obs',
    physics_steps=100,
    obs_steps=200,
):
    """
    Sliding-window / cycling 4D-Var.
    Ported from SlidingWindow_PyTorch.ipynb
    """
    N, T_total, _ = Y_long.shape
    starts = list(range(0, T_total - window_T + 1, stride))

    analyses = []
    backgrounds = []
    histories = []

    xb = None

    for c, start in enumerate(starts):
        Y_win = Y_long[:, start:start + window_T]

        if c == 0:
            if init_mode == 'baseline':
                xb = baseline_init_l96(dyn_sys, Y_win)
            else:
                raise ValueError(f"Unsupported init_mode: {init_mode}")

        X0_opt, hist = run_4dvar_l96(
            dyn_sys=dyn_sys,
            inverter=inverter,
            corr=corr,
            X0_init=xb,
            Y=Y_win,
            T=window_T,
            sigma_b=sigma_b,
            sigma_obs=sigma_obs,
            sigma_p=sigma_p,
            mode=opt_mode,
            physics_steps=physics_steps,
            obs_steps=obs_steps,
        )

        analyses.append(X0_opt.detach())
        backgrounds.append(xb.detach())
        histories.append(hist)

        # Forecast analysis forward to become next window background
        if start != starts[-1]:
            xb = dyn_sys.integrate(X0_opt.detach(), stride + 1)[-1].detach()

    return {
        'starts': starts,
        'analyses': torch.stack(analyses, dim=1),      # (N, n_cycles, grid)
        'backgrounds': torch.stack(backgrounds, dim=1), # (N, n_cycles, grid)
        'histories': histories,
    }

## 2. System Initialization
We setup the Lorenz96 system and estimate the spatial correlation structure used for preconditioning.

In [ ]:
print("--- Setup Lorenz96 System ---")
L96 = Lorenz96(grid_size=40, F=8.0, dt=0.01, observe_every=4, n_inner=10)

print("--- Estimating Correlation ---")
corr = estimate_correlation(L96, n_samples=1000, n_warmup=1000)

print("--- Generating Truth Trajectory ---")
T_total = 30
sigma_obs = 0.5
X0_gt, X_true_traj, Y, Y_clean = generate_data(
    L96, n_samples=1, n_time_steps=T_total, 
    n_warmup=500, obs_noise_std=sigma_obs, seed=777
)

## 3. Data Assimilation Experiments

In [ ]:
print("--- Running Full-Window 4D-Var ---")
X0_init_full = baseline_init_l96(L96, Y)
X0_opt_full, _ = run_4dvar_l96(
    L96, None, corr, X0_init_full, Y, T=T_total,
    sigma_b=1.0, sigma_obs=sigma_obs, mode='obs', obs_steps=500
)
X_opt_full_traj = L96.integrate(X0_opt_full, T_total)

print("--- Running Sliding-Window 4D-Var ---")
window_T = 10
stride = 1
res_sw = run_sliding_window_4dvar_l96(
    L96, None, corr, Y, window_T=window_T, stride=stride,
    sigma_b=0.3, sigma_obs=sigma_obs, init_mode='baseline', opt_mode='obs'
)

# Reconstruction
sw_traj_list = []
for i, start in enumerate(res_sw['starts']):
    n_steps = stride if i < len(res_sw['starts']) - 1 else T_total - start
    segment = L96.integrate(res_sw['analyses'][:, i], n_steps)
    sw_traj_list.append(segment)
X_opt_sw_traj = torch.cat(sw_traj_list, dim=0)

## 4. Results Visualization

In [ ]:
t = np.arange(T_total)

plt.figure(figsize=(15, 6))
plt.plot(t, X_true_traj[0, :, 0].cpu(), 'k-', label='Truth', alpha=0.3, lw=1)
plt.plot(t, X_opt_full_traj[:, 0, 0].cpu(), 'r--', label='Full-Window 4D-Var', alpha=0.9, lw=3)
plt.plot(t, X_opt_sw_traj[:, 0, 0].cpu(), 'b-', label='Sliding-Window 4D-Var', alpha=0.8, lw=1.5)
plt.scatter(t[::2], Y[0, ::2, 0].cpu(), c='g', s=20, label='Observations', alpha=0.4, edgecolors='k')
plt.title("Lorenz96 Trajectory Tracking ($X_0$) - Full vs Sliding Window")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.2)
plt.show()

# Instantaneous Error
T_min = min(X_opt_full_traj.shape[0], X_true_traj.shape[1], X_opt_sw_traj.shape[0])
err_full = torch.norm(X_opt_full_traj[:T_min, 0, :] - X_true_traj[0, :T_min, :], dim=-1)
err_sw = torch.norm(X_opt_sw_traj[:T_min, 0, :] - X_true_traj[0, :T_min, :], dim=-1)

plt.figure(figsize=(12, 5))
plt.plot(t[:T_min], err_full.cpu(), 'r-', label='Full-Window Error')
plt.plot(t[:T_min], err_sw.cpu(), 'b-', label='Sliding-Window Error')
plt.yscale('log')
plt.title("Instantaneous State Error (L2 Norm)")
plt.xlabel("Time Step")
plt.ylabel("Error")
plt.legend()
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()